In [ ]:
import dask.dataframe as dd
import pandas as pd
import os
folder_path = r'..\data\telefonia' 

ddf = dd.read_parquet(folder_path)

usuarios_counts = ddf.groupby('hashed_imsi_CENIA').size().compute().reset_index(name='cantidad')
usuarios_counts_sorted = usuarios_counts.sort_values('cantidad', ascending=False).reset_index(drop=True)

# print(f"Usuarios únicos: {usuarios_counts['hashed_imsi_CENIA'].nunique()}")
# print(f"Usuarios con más registros: {usuarios_counts_sorted.head(20)}")

# Eliminar el usuario con más registros para evitar sesgos en el análisis
top_user_id = usuarios_counts_sorted.iloc[0]['hashed_imsi_CENIA']
df_telefonia = ddf[ddf['hashed_imsi_CENIA'] != top_user_id]

# print(df_telefonia.head(5))

# Separar fecha y hora en columnas distintas 
df_telefonia['end_absolutetime_ts'] = dd.to_datetime(df_telefonia['end_absolutetime_ts'], errors='coerce')
df_telefonia = df_telefonia.dropna(subset=['end_absolutetime_ts'])
df_telefonia['fecha'] = df_telefonia['end_absolutetime_ts'].dt.date
df_telefonia['hora'] = df_telefonia['end_absolutetime_ts'].dt.time


In [ ]:
import dask.dataframe as dd

# Filtro: quedarse solo con días que tienen suficientes pings
MIN_REGISTROS_DIA = 5   # ajustable

# Contar pings por usuario-dia
conteos = (
    df_telefonia
    .groupby(['hashed_imsi_CENIA', 'fecha'])
    .size()
    .reset_index()
    .rename(columns={0: 'n_pings'})
    .compute()
)

dias_ok = conteos[conteos['n_pings'] >= MIN_REGISTROS_DIA][['hashed_imsi_CENIA', 'fecha']]

print(f"Umbral mínimo de pings por día : {MIN_REGISTROS_DIA}")
print(f"Días válidos (usuario × día)   : {len(dias_ok):,}")
print(f"Usuarios con ≥1 día válido     : {dias_ok['hashed_imsi_CENIA'].nunique():,}")
print()
print("Distribución de días válidos por usuario:")
print(dias_ok.groupby('hashed_imsi_CENIA').size().describe().round(1))

usuarios_validos = set(dias_ok['hashed_imsi_CENIA'].unique())
df_filtrado = df_telefonia[df_telefonia['hashed_imsi_CENIA'].isin(usuarios_validos)]
print(f"Registros originales: {len(df_telefonia):,}")
print(f"\nRegistros después del filtrado: {len(df_filtrado):,}")



Umbral mínimo de pings por día : 5
Días válidos (usuario × día)   : 33,762
Usuarios con ≥1 día válido     : 16,364

Distribución de días válidos por usuario:
count    16364.0
mean         2.1
std          2.9
min          1.0
25%          1.0
50%          1.0
75%          2.0
max         30.0
dtype: float64
Registros originales: 22,145,965

Registros después del filtrado: 556,330


In [ ]:
import pandas as pd
import numpy as np

# Cargar datos filtrados 
df = df_filtrado.compute()  # convertir a pandas para análisis local

# Ordenar por usuario y tiempo para facilitar el cálculo de distancias y tiempos entre pings consecutivos
df = df.sort_values(['hashed_imsi_CENIA', 'end_absolutetime_ts']).reset_index(drop=True)

# Calcular distancia y tiempo entre pings consecutivos del mismo usuario/día
def haversine_vectorized(lat1, lon1, lat2, lon2):
    #Distancia en metros entre dos puntos (lat/lon) usando fórmula haversine
    R = 6_371_000  # radio tierra en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Columnas del ping anterior dentro del mismo (usuario, día)
g = df.groupby(['hashed_imsi_CENIA', 'fecha'])
df['lat_prev'] = g['end_geo_lat'].shift(1)
df['lon_prev'] = g['end_geo_lon'].shift(1)
df['ts_prev']  = g['end_absolutetime_ts'].shift(1)

# Solo filas con ping anterior 
df_consec = df.dropna(subset=['lat_prev']).copy()

df_consec['distancia_m'] = haversine_vectorized(
    df_consec['lat_prev'], df_consec['lon_prev'],
    df_consec['end_geo_lat'], df_consec['end_geo_lon']
)
df_consec['delta_min'] = (
    df_consec['end_absolutetime_ts'] - df_consec['ts_prev']
).dt.total_seconds() / 60

# Velocidad implícita (m/s) entre pings consecutivos, evitando división por cero
df_consec['velocidad_ms'] = df_consec['distancia_m'] / (df_consec['delta_min'] * 60 + 1e-9)

# Resumen estadístico
print("=== Distancia entre pings consecutivos (metros) ===")
print(df_consec['distancia_m'].describe(percentiles=[.25,.5,.75,.9,.99]).round(1))
print()
print("=== Tiempo entre pings consecutivos (minutos) ===")
print(df_consec['delta_min'].describe(percentiles=[.25,.5,.75,.9,.99]).round(1))
print()
print("=== Velocidad implícita (m/s) ===")
print(df_consec['velocidad_ms'].describe(percentiles=[.25,.5,.75,.9,.99]).round(2))

# Cuántos usuarios realmente se movieron?
# "Se movió" = al menos un par de pings con distancia > 200m en el mismo día
se_movio = (
    df_consec[df_consec['distancia_m'] > 200]
    .groupby('hashed_imsi_CENIA')['fecha']
    .nunique()
    .reset_index(name='dias_con_movimiento')
)
total_usuarios = df['hashed_imsi_CENIA'].nunique()
print(f"\n=== Usuarios con al menos 1 día con movimiento (>200m entre pings) ===")
print(f"{len(se_movio):,} de {total_usuarios:,} usuarios ({100*len(se_movio)/total_usuarios:.1f}%)")


=== Distancia entre pings consecutivos (metros) ===
count     343297.0
mean        1824.5
std        26833.5
min            0.0
25%           67.5
50%          178.0
75%          626.1
90%         2332.2
99%        16243.2
max      1884410.4
Name: distancia_m, dtype: float64

=== Tiempo entre pings consecutivos (minutos) ===
count    343297.0
mean        150.8
std         167.5
min           0.0
25%          29.4
50%          89.1
75%         213.2
90%         390.4
99%         740.3
max        1235.3
Name: delta_min, dtype: float64

=== Velocidad implícita (m/s) ===
count    3.432970e+05
mean     1.351998e+08
std      2.503118e+10
min      0.000000e+00
25%      1.000000e-02
50%      4.000000e-02
75%      2.300000e-01
90%      1.060000e+00
99%      1.856000e+01
max      1.196049e+13
Name: velocidad_ms, dtype: float64

=== Usuarios con al menos 1 día con movimiento (>200m entre pings) ===
14,185 de 16,364 usuarios (86.7%)


In [ ]:
# Filtrar por RM
LAT_MIN, LAT_MAX = -34.3, -33.0
LON_MIN, LON_MAX = -71.1, -70.2

dentro_rm = (
    (df['end_geo_lat'].between(LAT_MIN, LAT_MAX)) &
    (df['end_geo_lon'].between(LON_MIN, LON_MAX))
)
fraccion_rm = (
    df.assign(en_rm=dentro_rm)
    .groupby('hashed_imsi_CENIA')['en_rm']
    .mean()
)
UMBRAL_RM = 0.80
usuarios_rm = fraccion_rm[fraccion_rm >= UMBRAL_RM].index
df = df[df['hashed_imsi_CENIA'].isin(usuarios_rm)].copy()

print(f"Filtro RM (>= {UMBRAL_RM:.0%} pings en RM)")
print(f"  Usuarios antes : {fraccion_rm.index.nunique():,}")
print(f"  Usuarios despues: {df['hashed_imsi_CENIA'].nunique():,}")

# Filtrar por días con suficientes pings 
MIN_PINGS_DIA = 5

pings_x_dia = df.groupby(['hashed_imsi_CENIA', 'fecha']).size().rename('n_pings')
dias_buenos = pings_x_dia[pings_x_dia >= MIN_PINGS_DIA].reset_index()[['hashed_imsi_CENIA', 'fecha']]

# Filtrar df a solo esos dias
df = df.merge(dias_buenos, on=['hashed_imsi_CENIA', 'fecha'], how='inner')

print(f"\nFiltro dias con >= {MIN_PINGS_DIA} pings")
print(f"  Dias-usuario antes : {len(pings_x_dia):,}")
print(f"  Dias-usuario despues: {len(dias_buenos):,}")
print(f"  Usuarios restantes  : {df['hashed_imsi_CENIA'].nunique():,}")
print(f"  Registros restantes : {len(df):,}")

Filtro RM (>= 80% pings en RM)
  Usuarios antes : 16,364
  Usuarios despues: 6,744

Filtro dias con >= 5 pings
  Dias-usuario antes : 91,886
  Dias-usuario despues: 13,507
  Usuarios restantes  : 6,744
  Registros restantes : 96,751


In [ ]:
# Distribución de pings por usuario-dia después de filtros
pings_x_dia = (
    df.groupby(['hashed_imsi_CENIA', 'fecha'])
    .size()
    .rename('n_pings')
)

print("=== Pings por usuario-dia ===")
print(pings_x_dia.describe(percentiles=[.25, .5, .75, .9, .95]).round(1))
print()

# Cuantos dias tienen suficientes pings para ser utiles
for umbral in [5, 8, 10, 15, 20]:
    n = (pings_x_dia >= umbral).sum()
    pct = 100 * n / len(pings_x_dia)
    print(f"  Dias con >= {umbral:2d} pings: {n:,}  ({pct:.1f}%)")

print()
# Cuantos usuarios tienen AL MENOS UN dia con bastantes pings
for umbral in [8, 10, 15]:
    usuarios = pings_x_dia[pings_x_dia >= umbral].index.get_level_values(0).nunique()
    pct = 100 * usuarios / df['hashed_imsi_CENIA'].nunique()
    print(f"  Usuarios con >= 1 dia de {umbral}+ pings: {usuarios:,}  ({pct:.1f}%)")

=== Pings por usuario-dia ===
count    13507.0
mean         7.2
std          4.5
min          5.0
25%          5.0
50%          6.0
75%          7.0
90%         11.0
95%         15.0
max         86.0
Name: n_pings, dtype: float64

  Dias con >=  5 pings: 13,507  (100.0%)
  Dias con >=  8 pings: 3,059  (22.6%)
  Dias con >= 10 pings: 1,823  (13.5%)
  Dias con >= 15 pings: 774  (5.7%)
  Dias con >= 20 pings: 393  (2.9%)

  Usuarios con >= 1 dia de 8+ pings: 986  (14.6%)
  Usuarios con >= 1 dia de 10+ pings: 523  (7.8%)
  Usuarios con >= 1 dia de 15+ pings: 203  (3.0%)


In [ ]:
import pandas as pd
stops = pd.read_csv('../data/gtfs/stops.txt')
viajes = pd.read_csv('../data/viajes/viajes_limpios_muestra.csv', sep=';')

print("Ejemplo stop_id GTFS:", stops['stop_id'].iloc[0])
print("Ejemplo paradero viajes:", viajes['paradero_subida_1'].iloc[0])

# Ver cuántos paraderos de viajes están en el GTFS
match = viajes['paradero_subida_1'].isin(stops['stop_id'])
print(f"\nParaderos que hacen match: {match.sum()} de {len(viajes)} ({100*match.mean():.1f}%)")


Ejemplo stop_id GTFS: PD1641
Ejemplo paradero viajes: L-26-12-30-PO

Paraderos que hacen match: 0 de 5000 (0.0%)


In [ ]:
import folium
import numpy as np

#Elegir usuario que se mueva harto en total
def bbox_diag_km(grp):
    dlat = grp['end_geo_lat'].max() - grp['end_geo_lat'].min()
    dlon = grp['end_geo_lon'].max() - grp['end_geo_lon'].min()
    # aprox en km (1 grado lat ~ 111km, 1 grado lon ~ 85km en Santiago)
    return np.sqrt((dlat * 111) ** 2 + (dlon * 85) ** 2)

movilidad = (
    df.groupby('hashed_imsi_CENIA')
    .apply(bbox_diag_km)
    .rename('diag_km')
    .reset_index()
)

# Usuarios que se mueven mas de 10km en total y tienen varios dias
resumen = (
    df.groupby('hashed_imsi_CENIA')
    .agg(n_dias=('fecha', 'nunique'), n_pings=('end_absolutetime_ts', 'count'))
    .reset_index()
    .merge(movilidad, on='hashed_imsi_CENIA')
    .query('diag_km >= 10 and n_dias >= 3')
    .sort_values('diag_km', ascending=False)
)

print(f"Usuarios con movilidad >= 10km y >= 3 dias: {len(resumen):,}")
print(resumen[['diag_km','n_dias','n_pings']].head(10).round(1).to_string())

usuario_id = resumen.iloc[0]['hashed_imsi_CENIA']
print(f"\nUsuario seleccionado (mayor movilidad): {usuario_id}")
print(f"  Diagonal bbox: {resumen.iloc[0]['diag_km']:.1f} km")
print(f"  Dias: {resumen.iloc[0]['n_dias']}")
print(f"  Pings totales: {resumen.iloc[0]['n_pings']}")

# Filtrar datos del usuario seleccionado, ordenados por tiempo y con coordenadas válidas
df_u = (
    df[df['hashed_imsi_CENIA'] == usuario_id]
    .sort_values('end_absolutetime_ts')
    .dropna(subset=['end_geo_lat', 'end_geo_lon'])
)

# Crear mapa
centro = [df_u['end_geo_lat'].mean(), df_u['end_geo_lon'].mean()]
mapa = folium.Map(location=centro, zoom_start=12, tiles='CartoDB positron')

colores = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
           '#a65628','#f781bf','#999999','#00ced1','#8b0000']
dias = sorted(df_u['fecha'].unique())
color_por_dia = {dia: colores[i % len(colores)] for i, dia in enumerate(dias)}

for dia in dias:
    df_dia = df_u[df_u['fecha'] == dia].reset_index(drop=True)
    color = color_por_dia[dia]
    coords = list(zip(df_dia['end_geo_lat'], df_dia['end_geo_lon']))

    if len(coords) >= 2:
        folium.PolyLine(coords, color=color, weight=2.5, opacity=0.8,
                        tooltip=str(dia)).add_to(mapa)

    for _, row in df_dia.iterrows():
        folium.CircleMarker(
            location=[row['end_geo_lat'], row['end_geo_lon']],
            radius=4, color=color, fill=True, fill_opacity=0.9,
            tooltip=f"{dia} {row['hora']}"
        ).add_to(mapa)

import os; os.makedirs('../outputs', exist_ok=True)
mapa.save('../outputs/mapa_usuario.html')
print(f"\nMapa guardado en outputs/mapa_usuario.html")
mapa


Usuarios con movilidad >= 10km y >= 3 dias: 34
      diag_km  n_dias  n_pings
4723    115.7       3       20
2117     50.0      10       58
3955     41.5      25      367
4833     38.9      15      121
887      34.5      11       77
3687     21.5      11       76
3608     19.5       6       47
5939     18.8      21      479
1329     18.5       3       64
6246     17.9      11       67

Usuario seleccionado (mayor movilidad): b27989611740ef0876afc0143e70690f399f41e66c17205d5d400df597c1a710
  Diagonal bbox: 115.7 km
  Dias: 3
  Pings totales: 20

Mapa guardado en outputs/mapa_usuario.html


In [ ]:
import pandas as pd
from pyproj import Transformer
import unicodedata

def normalizar(texto):
    texto = str(texto).upper().strip()
    texto = unicodedata.normalize('NFD', texto)
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

# Fuente 1 Buses: consolidado DTPM (codigos L-, T-, E-)
df_dtpm = pd.read_excel('../data/gtfs/paradas/2026-03-21_consolidado_Registro-Paradas_anual.xlsx')
df_dtpm.columns = ['orden','codigo_ts','codigo_usuario','sentido','variante','un',
                   'codigo_paradero','codigo_paradero_usuario','comuna','eje',
                   'desde','hacia','x','y','nombre','zona_paga','excepciones']
df_dtpm['x'] = pd.to_numeric(df_dtpm['x'], errors='coerce')
df_dtpm['y'] = pd.to_numeric(df_dtpm['y'], errors='coerce')
df_dtpm = df_dtpm.dropna(subset=['x','y','codigo_paradero']).copy()

transformer = Transformer.from_crs('EPSG:32719', 'EPSG:4326', always_xy=True)
lon, lat = transformer.transform(df_dtpm['x'].values, df_dtpm['y'].values)
df_dtpm['lat'] = lat
df_dtpm['lon'] = lon

buses = (
    df_dtpm.groupby('codigo_paradero')
    .agg(lat=('lat','first'), lon=('lon','first'), nombre=('nombre','first'))
    .reset_index()
    .rename(columns={'codigo_paradero': 'paradero'})
)
print(f"Paraderos de bus  : {len(buses):,}")

# Fuente 2 Metro + Metrotren: estaciones desde GTFS
trips      = pd.read_csv('../data/gtfs/trips.txt')
stop_times = pd.read_csv('../data/gtfs/stop_times.txt')
stops      = pd.read_csv('../data/gtfs/stops.txt')

lineas_metro = ['L1','L2','L3','L4','L4A','L5','L6','MTR','MTN']
trips_metro  = trips[trips['route_id'].astype(str).isin(lineas_metro)]

stops_metro = (
    stop_times[stop_times['trip_id'].isin(trips_metro['trip_id'])]
    .merge(stops[['stop_id','stop_name','stop_lat','stop_lon']], on='stop_id')
    [['stop_id','stop_name','stop_lat','stop_lon']]
    .drop_duplicates('stop_id')
    .copy()
)

# Limpiar nombres:
# 1) cortar antes de "Dirección" (metro L1-L6)
# 2) cortar antes de "(Anden" (metrotren MTR/MTN)
stops_metro['paradero'] = (
    stops_metro['stop_name']
    .str.split(r'[Dd]irecci', regex=True).str[0]
    .str.split(r'\(Anden',   regex=True).str[0]
    .str.strip()
    .apply(normalizar)
)

metro = (
    stops_metro[['paradero','stop_lat','stop_lon','stop_name']]
    .rename(columns={'stop_lat':'lat','stop_lon':'lon','stop_name':'nombre'})
    .drop_duplicates('paradero')
)
print(f"Estaciones metro/metrotren: {len(metro):,}")

# Aliases manuales para paraderos sin match exacto entre viajes y GTFS
aliases = {
    'CAL Y CANTO'          : 'PUENTE CAL Y CANTO',
    'PARQUE OHIGGINS'      : "PARQUE O'HIGGINS",
    'RONDIZONNI'           : 'RONDIZZONI',
    'PLAZA MAIPU'          : 'PLAZA DE MAIPU',
    'ESTACION ALAMEDA'     : 'ESTACION CENTRAL',
    'UNION LATINO AMERICANA': 'UNION LATINOAMERICANA',
}

# Construir filas extra para cada alias
alias_rows = []
coords_dict = pd.concat([buses, metro]).drop_duplicates('paradero').set_index('paradero')
for nombre_viaje, nombre_gtfs in aliases.items():
    if nombre_gtfs in coords_dict.index:
        row = coords_dict.loc[nombre_gtfs].copy()
        row.name = nombre_viaje
        alias_rows.append(row.reset_index().rename(columns={'index':'paradero'}).assign(paradero=nombre_viaje))
    else:
        print(f"  AVISO: '{nombre_gtfs}' no encontrado en diccionario")

if alias_rows:
    aliases_df = pd.DataFrame([{'paradero': k,
                                  'lat': coords_dict.loc[v,'lat'],
                                  'lon': coords_dict.loc[v,'lon'],
                                  'nombre': v}
                                 for k,v in aliases.items() if v in coords_dict.index])
else:
    aliases_df = pd.DataFrame(columns=['paradero','lat','lon','nombre'])

print(f"Aliases manuales  : {len(aliases_df):,}")

# Combinar y guardar
paraderos_coords = (
    pd.concat([buses, metro, aliases_df], ignore_index=True)
    .drop_duplicates('paradero')
)
paraderos_coords.to_csv('../data/gtfs/paradas/paraderos_coords.csv', index=False)
print(f"\nTotal paraderos en diccionario: {len(paraderos_coords):,}")

# Verificar cuántos paraderos de viajes están en el diccionario combinado
viajes = pd.read_csv('../data/viajes/viajes_limpios_muestra.csv', sep=';')
paraderos_set = set(paraderos_coords['paradero'])

viajes['sub_norm'] = viajes['paradero_subida_1'].apply(lambda x: normalizar(x) if pd.notna(x) else x)
viajes['baj_norm'] = viajes['paradero_bajada_1'].apply(lambda x: normalizar(x) if pd.notna(x) else x)

match_sub = viajes['sub_norm'].isin(paraderos_set)
match_baj = viajes['baj_norm'].isin(paraderos_set)
print(f"Match paradero_subida_1: {match_sub.sum():,} de {match_sub.notna().sum():,} ({100*match_sub.mean():.1f}%)")
print(f"Match paradero_bajada_1: {match_baj.sum():,} de {match_baj.notna().sum():,} ({100*match_baj.mean():.1f}%)")
print("\nParaderos sin match (top 10):")
print(viajes.loc[~match_sub, 'paradero_subida_1'].value_counts().head(10))

In [26]:
import pandas as pd
import unicodedata
import glob
import os

def normalizar(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

columnas_a_eliminar = [
    "Unnamed: 100",
    "mediahora_inicio_viaje", "mediahora_fin_viaje",
    "mediahora_bajada_1", "mediahora_bajada_2",
    "mediahora_bajada_3", "mediahora_bajada_4",
    "mediahora_inicio_viaje_hora", "mediahora_fin_viaje_hora",
    "op_1era_etapa", "op_2da_etapa", "op_3era_etapa", "op_4ta_etapa",
    "tv3", "tc3", "tv4", "tviaje", "tviaje2", "egreso",
    "proposito", "tv1", "tc1", "te1", "tv2", "tc2", "te2", "te3"
]

paraderos = pd.read_csv("../data/gtfs/paradas/paraderos_coords.csv")[["paradero","lat","lon"]].drop_duplicates("paradero")
sub_coords = paraderos.rename(columns={"paradero":"sub_norm", "lat":"lat_subida", "lon":"lon_subida"})
baj_coords = paraderos.rename(columns={"paradero":"baj_norm", "lat":"lat_bajada", "lon":"lon_bajada"})

os.makedirs("../data/viajes/parquet", exist_ok=True)

archivos = sorted(glob.glob("../data/viajes/*.viajes.csv.gz"))
print(f"Archivos a convertir: {len(archivos)}")
for a in archivos:
    print(f"  {os.path.basename(a)}  ({os.path.getsize(a)/1e6:.0f} MB gz)")

for archivo in archivos:
    fecha = os.path.basename(archivo)[:10]
    out = f"../data/viajes/parquet/{fecha}.parquet"
    if os.path.exists(out):
        print(f"  {fecha}: ya existe, saltando", flush=True)
        continue
    mb = os.path.getsize(archivo)/1e6
    print(f"Convirtiendo {fecha} ({mb:.0f} MB gz)...", flush=True)
    df = pd.read_csv(archivo, compression="gzip", sep="|")
    print(f"  Leido: {len(df):,} filas. Limpiando...", flush=True)
    cols_drop = [c for c in columnas_a_eliminar if c in df.columns]
    df = df.drop(columns=cols_drop)
    df["sub_norm"] = df["paradero_subida_1"].apply(lambda x: normalizar(x) if pd.notna(x) else x)
    df["baj_norm"] = df["paradero_bajada_1"].apply(lambda x: normalizar(x) if pd.notna(x) else x)
    df = df.merge(sub_coords, on="sub_norm", how="left")
    df = df.merge(baj_coords, on="baj_norm", how="left")
    df = df.drop(columns=["sub_norm", "baj_norm"])
    df.to_parquet(out, index=False, engine="pyarrow")
    tam_out = os.path.getsize(out)/1e6
    print(f"  Guardado: {fecha}.parquet  ({tam_out:.0f} MB)  subida {df['lat_subida'].notna().mean()*100:.1f}%", flush=True)

print("Conversion completa! Archivos en data/viajes/parquet/")
print('Lectura futura: dd.read_parquet("../data/viajes/parquet/")')

Archivos a convertir: 7
  2023-11-01.viajes.csv.gz  (107 MB gz)
  2023-11-02.viajes.csv.gz  (325 MB gz)
  2023-11-03.viajes.csv.gz  (333 MB gz)
  2023-11-04.viajes.csv.gz  (181 MB gz)
  2023-11-05.viajes.csv.gz  (99 MB gz)
  2023-11-06.viajes.csv.gz  (336 MB gz)
  2023-11-07.viajes.csv.gz  (339 MB gz)
Convirtiendo 2023-11-01 (107 MB gz)...
  Leido: 1,205,598 filas. Limpiando...
  Guardado: 2023-11-01.parquet  (96 MB)  subida 99.5%
Convirtiendo 2023-11-02 (325 MB gz)...
  Leido: 3,378,563 filas. Limpiando...
  Guardado: 2023-11-02.parquet  (282 MB)  subida 99.5%
Convirtiendo 2023-11-03 (333 MB gz)...
  Leido: 3,502,454 filas. Limpiando...
  Guardado: 2023-11-03.parquet  (291 MB)  subida 99.5%
Convirtiendo 2023-11-04 (181 MB gz)...
  Leido: 2,019,791 filas. Limpiando...
  Guardado: 2023-11-04.parquet  (160 MB)  subida 99.5%
Convirtiendo 2023-11-05 (99 MB gz)...
  Leido: 1,131,396 filas. Limpiando...
  Guardado: 2023-11-05.parquet  (88 MB)  subida 99.5%
Convirtiendo 2023-11-06 (336 MB gz)

In [28]:
import dask.dataframe as dd

ddf = dd.read_parquet("../data/viajes/parquet/")
print(f"Total viajes : {len(ddf):,}")
print(f"Columnas     : {len(ddf.columns)}")
print(f"Con subida   : {ddf['lat_subida'].notnull().sum().compute():,}")
print(f"Con bajada   : {ddf['lat_bajada'].notnull().sum().compute():,}")


Total viajes : 18,227,050
Columnas     : 78
Con subida   : 18,129,808
Con bajada   : 13,959,906
